# Data Acquisition for FinTech-AI-MIT Paper
## Downloads everything Colab can fetch automatically, lists the rest

**How to run:** Runtime → Run all. Takes ~15-30 minutes. Outputs land in
`/content/data/raw/` and a merged panel in `/content/data/processed/panel.parquet`.
Click the folder icon on the left to download files back to your laptop.

**What this gets you automatically:**
- World Bank WDI macro controls (growth, inflation, trade, etc.)
- World Bank country classifications (income groups, regions)
- Penn World Tables (human capital, real GDP)
- Global Findex 2021 wave (other waves below in manual section)
- OpenAlex AI publications by country-year (Stanford AI Index fallback)
- ILOSTAT employment by sector (for KIS share)

**What you'll still need to download manually** (one-time, links provided in
Section 9):
- Oxford GARI 2019-2024
- IMF AI Preparedness Index
- ILO Working Paper 140 country exposure
- GGDC 10-Sector database
- Older Findex waves (2011, 2014, 2017, 2024)
- Optional: Stanford AI Index official panel (request via Stanford HAI)

The notebook will work with whatever subset you have; missing sources just
get skipped with a warning.

## 1. Setup

In [2]:
!pip install -q wbdata country_converter pycountry pyarrow

import json, time, logging, os
from io import BytesIO
from pathlib import Path
from typing import Optional
import numpy as np
import pandas as pd
import requests
!pip install -q wbdata country_converter pycountry pyarrow openpyxl

# Working directories
RAW = Path("/content/data/raw"); RAW.mkdir(parents=True, exist_ok=True)
PROCESSED = Path("/content/data/processed"); PROCESSED.mkdir(parents=True, exist_ok=True)

logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s | %(levelname)s | %(message)s",
                    datefmt="%H:%M:%S")
log = logging.getLogger("data")

YEAR_MIN, YEAR_MAX = 2010, 2024
print("Setup complete. Outputs will land in /content/data/")

Setup complete. Outputs will land in /content/data/


In [3]:
from google.colab import drive
drive.mount('/content/drive')

# Then point your output paths into Drive instead of /content/
RAW = Path("/content/drive/MyDrive/fintech_ai_mit/data/raw")
PROCESSED = Path("/content/drive/MyDrive/fintech_ai_mit/data/processed")
RAW.mkdir(parents=True, exist_ok=True)
PROCESSED.mkdir(parents=True, exist_ok=True)

Mounted at /content/drive


## 2. Country code harmonization

We convert all sources to ISO 3166-1 alpha-3.

In [4]:
import country_converter as coco
import logging
logging.getLogger("country_converter").setLevel(logging.ERROR)

_CC = coco.CountryConverter()

MANUAL_ISO3 = {
    "kosovo": "XKX", "west bank and gaza": "PSE", "taiwan, china": "TWN",
    "korea, rep.": "KOR", "korea, dem. people's rep.": "PRK",
    "lao pdr": "LAO", "egypt, arab rep.": "EGY", "iran, islamic rep.": "IRN",
    "venezuela, rb": "VEN", "yemen, rep.": "YEM", "turkiye": "TUR",
    "cote d'ivoire": "CIV", "eswatini": "SWZ", "czechia": "CZE",
    "north macedonia": "MKD", "cabo verde": "CPV", "myanmar": "MMR",
    "congo, dem. rep.": "COD", "congo, rep.": "COG",
}

AGGREGATES = {
    "world", "high income", "low income", "middle income",
    "lower middle income", "upper middle income",
    "low & middle income", "least developed countries: un classification",
    "heavily indebted poor countries (hipc)",
    "ida total", "ida blend", "ida only", "ida & ibrd total",
    "ibrd only", "fragile and conflict affected situations",
    "east asia & pacific", "east asia & pacific (excluding high income)",
    "east asia & pacific (ida & ibrd countries)",
    "europe & central asia", "europe & central asia (excluding high income)",
    "europe & central asia (ida & ibrd countries)",
    "european union", "euro area",
    "latin america & caribbean", "latin america & caribbean (excluding high income)",
    "latin america & the caribbean (ida & ibrd countries)",
    "middle east & north africa", "middle east & north africa (excluding high income)",
    "middle east & north africa (ida & ibrd countries)",
    "middle east, north africa, afghanistan & pakistan",
    "north america",
    "south asia", "south asia (ida & ibrd)",
    "sub-saharan africa", "sub-saharan africa (excluding high income)",
    "sub-saharan africa (ida & ibrd countries)",
    "arab world",
    "central europe and the baltics",
    "caribbean small states", "pacific island small states",
    "other small states", "small states",
    "post-demographic dividend", "pre-demographic dividend",
    "early-demographic dividend", "late-demographic dividend",
    "oecd members",
    "channel islands", "not classified",
}

def is_aggregate(name_lower: str) -> bool:
    return name_lower in AGGREGATES

def to_iso3(names, source="?"):
    s = pd.Series(names).astype(str).str.strip()
    sl = s.str.lower()
    out = pd.Series([None]*len(s), index=s.index, dtype="object")

    # Drop aggregates
    out.loc[sl.apply(is_aggregate)] = None

    # Manual overrides
    mask = out.isna() & sl.isin(MANUAL_ISO3.keys())
    out.loc[mask] = sl.loc[mask].map(MANUAL_ISO3)

    # Everything else through country_converter
    rem = out.isna() & ~sl.apply(is_aggregate)
    if rem.any():
        r = _CC.convert(names=s.loc[rem].tolist(), to="ISO3", not_found=None)
        if isinstance(r, str):
            r = [r]
        # CRITICAL: coerce list returns (ambiguous matches) and "not found" to None
        r = [None if (isinstance(x, list) or x == "not found") else x for x in r]
        out.loc[rem] = r

    return out

print("Country code mapping ready (with list-coercion fix).")

Country code mapping ready (with list-coercion fix).


## 3. World Bank WDI — macro controls (automated)

In [5]:
import wbdata
from datetime import datetime

WDI_INDICATORS = {
    "NY.GDP.PCAP.KD.ZG": "gdp_pc_growth",
    "NY.GDP.PCAP.PP.KD": "gdp_pc_ppp",
    "NE.TRD.GNFS.ZS": "trade_pct_gdp",
    "FP.CPI.TOTL.ZG": "inflation",
    "NE.GDI.TOTL.ZS": "investment_pct_gdp",
    "SP.POP.GROW": "pop_growth",
    "FS.AST.PRVT.GD.ZS": "credit_to_private_pct_gdp",
    "SE.SEC.ENRR": "secondary_enrollment",
    "SE.TER.ENRR": "tertiary_enrollment",
}

log.info("Downloading WDI ({} indicators)...".format(len(WDI_INDICATORS)))
wdi = wbdata.get_dataframe(WDI_INDICATORS,
    date=(datetime(YEAR_MIN,1,1), datetime(YEAR_MAX,12,31))).reset_index()
wdi = wdi.rename(columns={"country":"country_name", "date":"year"})
wdi["year"] = wdi["year"].astype(str).str[:4].astype(int)
wdi["iso3"] = to_iso3(wdi["country_name"], "WDI")
wdi = wdi.dropna(subset=["iso3"]).drop(columns="country_name")
wdi = wdi[["iso3","year"] + list(WDI_INDICATORS.values())].sort_values(["iso3","year"])
wdi.to_csv(RAW/"wdi_controls.csv", index=False)
log.info(f"WDI saved: {len(wdi):,} rows, {wdi['iso3'].nunique()} countries")
wdi.head()

,iso3,year,gdp_pc_growth,gdp_pc_ppp,trade_pct_gdp,inflation,investment_pct_gdp,pop_growth,credit_to_private_pct_gdp,secondary_enrollment,tertiary_enrollment
884,ABW,2010,-2.956953,33951.624618,135.464655,2.078141,28.283180,0.230041,58.800657,98.220741,36.062531
883,ABW,2011,2.610525,34837.940287,154.437787,4.316297,27.416637,0.736689,56.250434,96.633919,37.864811
882,ABW,2012,-2.484648,33972.340108,156.645197,0.627472,23.522507,1.469782,58.685903,108.976189,38.864182
881,ABW,2013,4.855258,35621.784855,155.731904,-2.372065,23.825860,1.492031,59.163685,NaN,17.125870
880,ABW,2014,0.351128,35746.863068,156.491053,0.421441,23.111441,1.065512,60.240539,NaN,17.751900


## 4. World Bank income classifications (automated)

In [6]:
log.info("Downloading WB country classifications...")
r = requests.get("https://api.worldbank.org/v2/country?format=json&per_page=400", timeout=60)
data = r.json()
countries = data[1] if isinstance(data, list) and len(data) > 1 else []

inc = pd.DataFrame([{
    "iso3": c["id"],
    "country_name": c["name"],
    "income_group_label": c.get("incomeLevel", {}).get("value", None),
    "region": c.get("region", {}).get("value", None),
} for c in countries])
inc = inc[~inc["region"].isin(["Aggregates", None])]
inc["income_group"] = inc["income_group_label"].map({
    "Low income": "L", "Lower middle income": "LM",
    "Upper middle income": "UM", "High income": "H"
})
inc["mi_status"] = inc["income_group"].isin(["LM","UM"]).astype(int)
inc["hi_status"] = (inc["income_group"]=="H").astype(int)
inc.to_csv(RAW/"income_groups_current.csv", index=False)
log.info(f"Income groups: {len(inc)} countries")
print("\nNote: this gives only CURRENT classification. For historical")
print("year-by-year reclassification download CLASS.xlsx manually (see Section 9).")
inc.head()


Note: this gives only CURRENT classification. For historical
year-by-year reclassification download CLASS.xlsx manually (see Section 9).


,iso3,country_name,income_group_label,region,income_group,mi_status,hi_status
0,ABW,Aruba,High income,Latin America & Caribbean,H,0,1
2,AFG,Afghanistan,Low income,"Middle East, North Africa, Afghanistan & Pakistan",L,0,0
5,AGO,Angola,Lower middle income,Sub-Saharan Africa,LM,1,0
6,ALB,Albania,Upper middle income,Europe & Central Asia,UM,1,0
7,AND,Andorra,High income,Europe & Central Asia,H,0,1


## 5. Penn World Tables 10.01 (direct HTTP)

In [7]:
log.info("Downloading Penn World Tables 10.01...")
PWT_URL = "https://dataverse.nl/api/access/datafile/354095"  # PWT 10.01 xlsx (current stable)
# PWT_URL = "https://dataverse.nl/api/access/datafile/554105"  # PWT 11.0 xlsx (uncomment if/when released)
try:
    r = requests.get(PWT_URL, timeout=180,
                     headers={"User-Agent": "academic-research/1.0"})
    r.raise_for_status()
    raw = pd.read_excel(BytesIO(r.content), sheet_name="Data")
    keep = [c for c in ["countrycode","year","hc","rgdpna","pop","emp"] if c in raw.columns]
    pwt = raw[keep].rename(columns={"countrycode":"iso3", "hc":"human_capital_pwt"})
    pwt = pwt[(pwt["year"]>=YEAR_MIN) & (pwt["year"]<=YEAR_MAX)]
    pwt.to_csv(RAW/"pwt.csv", index=False)
    log.info(f"PWT saved: {len(pwt):,} rows, {pwt['iso3'].nunique()} countries")
    pwt.head()
except Exception as e:
    log.error(f"PWT download failed: {e}")
    print("Manual fallback: https://www.rug.nl/ggdc/productivity/pwt/")

## 6. Global Findex — 2021 wave (direct HTTP)

The 2021 wave has a stable URL. For 2011, 2014, 2017, and 2024 waves you need
to visit the Findex download page manually (see Section 9). The four-component
FinTech index can still be built with just one wave for prototyping.

In [8]:
# The 2025 release of the Findex database bundles ALL waves (2011, 2014, 2017,
# 2021, 2024) in a single country-level file. We download once and filter to 2021.
FINDEX_URL = ("https://thedocs.worldbank.org/en/doc/"
    "be6615202d1f08a25855c8ac2d615122-0050012025/related/GlobalFindexDatabase2025.csv")

log.info("Downloading Findex (2025 release, contains all waves)...")
try:
    r = requests.get(FINDEX_URL, timeout=180,
                     headers={"User-Agent": "academic-research/1.0"})
    r.raise_for_status()
    findex_all = pd.read_csv(BytesIO(r.content), low_memory=False)
    log.info(f"Full Findex file: {len(findex_all):,} rows, {len(findex_all.columns)} cols")

    # Find the year column (case-insensitive)
    year_col = next((c for c in findex_all.columns if c.lower() == "year"), None)
    if year_col is None:
        raise RuntimeError(f"No year column found. Columns: {list(findex_all.columns)[:20]}")

    print(f"\nWaves available in file: {sorted(findex_all[year_col].dropna().unique())}")

    # Filter to 2021 wave (change/duplicate this block if you want other waves)
    findex = findex_all[findex_all[year_col] == 2021].copy()
    findex["wave"] = 2021
    findex.to_csv(RAW/"findex_2021_raw.csv", index=False)
    log.info(f"Findex 2021 saved: {len(findex)} rows, {len(findex.columns)} cols")

    # Also save the full multi-wave file for later use
    findex_all.to_csv(RAW/"findex_all_waves_raw.csv", index=False)
    log.info(f"Findex all waves saved: {len(findex_all):,} rows")

    print("\nColumns containing 'fin' (candidate F-index components):")
    print([c for c in findex.columns if 'fin' in c.lower()][:20])
except Exception as e:
    log.error(f"Findex download failed: {e}")
    print("Manual fallback: https://www.worldbank.org/en/publication/globalfindex/download-data")



Waves available in file: [np.int64(2011), np.int64(2014), np.int64(2017), np.int64(2021), np.int64(2022), np.int64(2024)]

Columns containing 'fin' (candidate F-index components):
['fin4_d', 'fin11_2a', 'fin11a', 'fin11b', 'fin11c', 'fin11f', 'fin11d', 'fin11e', 'fin14a', 'fin14b', 'fin14c', 'fin14d', 'fin13_1a', 'fin13_1b', 'fin26a', 'fin26b', 'fin27a', 'fin27b', 'fin17f', 'fin17a_17a1_d']


## 7. OpenAlex — AI publications by country-year (Stanford AI Index fallback)

This is the FALLBACK for the Activity pillar of the AI index. When you receive
the Stanford AI Index official panel, replace the output of this section.

**SWAP-IN POINT:** if Stanford AI Index data arrives, save it as
`/content/data/raw/stanford_ai_index.csv` with columns
`iso3, year, ai_investment, ai_patents_count, ai_talent_share`. The
downstream G-index construction will pick it up automatically.

In [9]:
AI_CONCEPT = "C154945302"  # Artificial intelligence
years_to_query = range(2017, 2025)

log.info(f"Querying OpenAlex for AI publications, years {min(years_to_query)}-{max(years_to_query)}")
all_rows = []
for y in years_to_query:
    url = (f"https://api.openalex.org/works"
       f"?filter=concepts.id:{AI_CONCEPT},publication_year:{y},type:article"
       f"&group_by=authorships.institutions.country_code&per-page=200")
    try:
        d = requests.get(url, timeout=60).json()
        groups = d.get("group_by", [])
        for g in groups:
            all_rows.append({"country_iso2": g["key"].lower(),
                            "year": y, "ai_pubs_count": g["count"]})
        log.info(f"  year {y}: {len(groups)} countries")
        time.sleep(0.3)  # be polite to free API
    except Exception as e:
        log.error(f"  year {y}: {e}")

import pycountry
def iso2_to_iso3(c):
    try: return pycountry.countries.get(alpha_2=c.upper()).alpha_3
    except: return None

openalex = pd.DataFrame(all_rows)
if len(openalex) > 0:
    openalex["iso3"] = openalex["country_iso2"].apply(iso2_to_iso3)
    openalex = openalex.dropna(subset=["iso3"])[["iso3","year","ai_pubs_count"]]
    openalex.to_csv(RAW/"openalex_ai_pubs.csv", index=False)
    log.info(f"OpenAlex saved: {len(openalex):,} country-year rows")
openalex.head()

,iso3,year,ai_pubs_count


In [10]:
from pathlib import Path
RAW = Path("/content/data/raw")
for sub in ["oxford_gari", "wgi", "imf_aipi", "wb_income", "ilostat"]:
    (RAW/sub).mkdir(parents=True, exist_ok=True)
print("Folders ready:")
!ls -la /content/data/raw/

Folders ready:
total 28
drwxr-xr-x 7 root root 4096 Jun  9 04:22 .
drwxr-xr-x 4 root root 4096 Jun  9 04:20 ..
drwxr-xr-x 2 root root 4096 Jun  9 04:22 ilostat
drwxr-xr-x 2 root root 4096 Jun  9 04:22 imf_aipi
drwxr-xr-x 2 root root 4096 Jun  9 04:22 oxford_gari
drwxr-xr-x 2 root root 4096 Jun  9 04:22 wb_income
drwxr-xr-x 2 root root 4096 Jun  9 04:22 wgi


In [17]:
!find /content/data/raw -type f

/content/data/raw/ilo_rho_by_income.csv


## 7b. ILO WP 140 — GenAI occupational exposure (income-group aggregate)

The ILO Working Paper 140 publishes the country-level GenAI exposure shares only
as aggregates by **income group** and **region** (not by individual country). The
numbers below are transcribed from the interactive chart at
https://www.ilo.org/resource/article/how-might-generative-ai-impact-different-occupations
(Datawrapper id `noouS`, also reproduced in Table 8 of WP 140).

We merge these onto the panel via the WB income group. This means every High-income
country gets the same value (6.2% in Gradient 4), every UM gets 3.6%, etc. This is
a coarse but valid measure for **robustness checks** against a primary country-level
AI-exposure measure (e.g. IMF Cazzaniga 2024). It is collinear with income-group
fixed effects, so use it in specs without those FEs.


In [18]:
# ILO WP 140 income-group aggregates (Gradient 4 = highest GenAI exposure share, %)
# Source: ILO interactive chart, accessed June 2026.
ilo_rho = pd.DataFrame({
    "income_group":     ["H",  "UM", "LM", "L"],
    "rho_gradient4":    [6.2,  3.6,  1.9,  0.5],
    "rho_gradient3":    [10.9, 3.4,  1.9,  0.6],
    # TODO: fill in once you have the remaining columns visible
    # "rho_gradient2":  [...],
    # "rho_gradient1":  [...],
    # "rho_minimal":    [...],
    # "rho_not_exposed":[...],
})
ilo_rho["rho_exposed_top2"] = ilo_rho["rho_gradient4"] + ilo_rho["rho_gradient3"]
ilo_rho.to_csv(RAW/"ilo_rho_by_income.csv", index=False)
log.info(f"ILO WP140 income-group aggregates saved: {len(ilo_rho)} groups")
ilo_rho


,income_group,rho_gradient4,rho_gradient3,rho_exposed_top2
0,H,6.2,10.9,17.1
1,UM,3.6,3.4,7.0
2,LM,1.9,1.9,3.8
3,L,0.5,0.6,1.1


## 7c. Oxford GARI 2024 — Government AI Readiness Index

The 2024 GARI ranks 188 countries on AI readiness. Oxford Insights publishes the
scores as a PDF report (and on their site as a copyable table). To run this cell,
first save the Rank/Country/2024 Total columns as an xlsx at:
`/content/data/raw/oxford_gari/GARI_2024.xlsx`

If the file is missing, the cell skips with a warning so the rest of the
pipeline still runs.


In [22]:
GARI_DIR = RAW/"oxford_gari"
GARI_DIR.mkdir(parents=True, exist_ok=True)

gari_files = (list(GARI_DIR.glob("*GARI*.xlsx")) + list(GARI_DIR.glob("*GARI*.csv"))
              + list(GARI_DIR.glob("*gari*.xlsx")) + list(GARI_DIR.glob("*gari*.csv")))

if gari_files:
    fpath = gari_files[0]
    log.info(f"Reading GARI 2024 from {fpath}")

    if fpath.suffix.lower() == ".csv":
        try:
            with open(fpath, 'r', encoding='utf-8') as f:
                raw_lines = f.readlines()
        except UnicodeDecodeError:
            log.warning(f"  UnicodeDecodeError reading {fpath}, trying latin1.")
            with open(fpath, 'r', encoding='latin1') as f:
                raw_lines = f.readlines()

        if not raw_lines:
            raise ValueError(f"CSV file {fpath} is empty.")

        header_line = raw_lines[0].strip()
        expected_columns = [col.strip() for col in header_line.split(',')]
        normalized_headers = [str(c).strip().lower().replace(" ", "_") for c in expected_columns]

        try:
            # Try reading with pandas, skipping the header and assigning names manually
            gari = pd.read_csv(fpath, sep=',', header=None, skiprows=1)
            gari.columns = normalized_headers
            log.info(f"  CSV read successfully with explicit column assignment.")
        except Exception as e:
            log.warning(f"  pd.read_csv with explicit column assignment failed: {e}. Falling back to manual splitting.")
            # Fallback: manual splitting of data rows
            from io import StringIO
            data_string = "\n".join(raw_lines[1:]) # Re-create string from data lines
            parsed_data = []
            for line in StringIO(data_string):
                if line.strip(): # Avoid empty lines
                    parsed_data.append([item.strip() for item in line.split(',')])
            gari = pd.DataFrame(parsed_data, columns=normalized_headers)
            log.info(f"  Manually parsed CSV into {len(normalized_headers)} columns.")

    else: # Excel file handling
        gari = pd.read_excel(fpath)
        # Normalize column names for Excel as well
        gari.columns = [str(c).strip().lower().replace(" ", "_") for c in gari.columns]

    log.info(f"  columns after parsing: {list(gari.columns)}")

    # Now, with correctly parsed columns, the score_col and country_col detection should work
    score_col   = next((c for c in gari.columns if "total" in c or "score" in c), None)
    country_col = next((c for c in gari.columns if "country" in c or "economy" in c), None)

    if score_col is None or country_col is None:
        raise RuntimeError(f"Could not find country/score columns even after robust parsing. Got: {list(gari.columns)}")

    # The rename step will now correctly map `total` to `gari_total`
    gari = gari.rename(columns={score_col: "gari_total", country_col: "country"})
    gari["iso3"] = to_iso3(gari["country"], source="GARI")
    gari = gari.dropna(subset=["iso3"]).copy()
    gari["year"] = 2024
    gari = (gari[["iso3", "year", "gari_total"]]
            .drop_duplicates(subset=["iso3","year"]))
    gari["gari_total"] = pd.to_numeric(gari["gari_total"], errors="coerce")
    gari.to_csv(RAW/"gari_2024.csv", index=False)
    log.info(f"GARI saved: {len(gari)} countries with iso3")
    print(gari.head())
else:
    log.warning(f"No GARI file found in {GARI_DIR}. "
                f"Place GARI_2024.xlsx or GARI_2024.csv there.")

In [21]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [20]:
import os
p = "/content/data/raw/oxford_gari"
print(os.path.exists(p))
print(os.listdir(p) if os.path.exists(p) else "dir missing")

True
[]


## 7d. Worldwide Governance Indicators (WGI)

Download `wgidataset.xlsx` from
https://www.worldbank.org/en/publication/worldwide-governance-indicators/interactive-data-access
and save it to `/content/data/raw/wgi/wgidataset.xlsx`.

The file is one row per country-year-indicator. We reshape it so each of the
six WGI indicators becomes its own column on the country-year panel. We grab
both the z-score `estimate` (-2.5 to +2.5, the classic scale) and the
`percentile_rank` (0-100, easier to interpret). The cell skips gracefully if
the file is missing so the rest of the notebook still runs.


In [14]:
WGI_PATH = RAW/"wgi/wgidataset.xlsx"

SHEET_TO_CODE = {
    "VoiceandAccountability":         "va",
    "Political StabilityNoViolence":  "pv",
    "GovernmentEffectiveness":        "ge",
    "RegulatoryQuality":              "rq",
    "RuleofLaw":                      "rl",
    "ControlofCorruption":            "cc",
}

def parse_wgi_sheet(path, sheet, ind_code):
    """WGI per-indicator sheet:
       Row N:   year labels (merged, repeated across sub-cols)
       Row N+1: 'Country/Territory | Code | Estimate | StdErr | NumSrc | Rank | Lower | Upper'  (repeated per year)
       Row N+2+: data
    """
    raw = pd.read_excel(path, sheet_name=sheet, header=None)

    # Find the REAL header row — must contain Country/Territory AND Code AND Estimate AND StdErr
    sub_row = None
    for i in range(min(25, len(raw))):
        cells = [str(v).strip().lower() for v in raw.iloc[i].tolist()]
        cell_set = set(cells)
        if ("estimate" in cell_set and "stderr" in cell_set
                and any("country" in c for c in cells)
                and "code" in cell_set):
            sub_row = i
            break
    if sub_row is None:
        log.warning(f"  {sheet}: real header row not found; skipping")
        return None

    years = raw.iloc[sub_row - 1].ffill()
    subs  = raw.iloc[sub_row]

    new_cols = []
    for y, s in zip(years, subs):
        try:
            yi = int(float(y))
        except (ValueError, TypeError):
            yi = None
        new_cols.append((yi, str(s).strip().lower() if pd.notna(s) else ""))

    data = raw.iloc[sub_row + 1:].copy()
    data.columns = new_cols

    # Locate country and code columns by their labels (year is None for these)
    country_col = next((c for c in new_cols if c[1].startswith("country")), None)
    code_col    = next((c for c in new_cols if c[1] == "code"), None)
    if country_col is None or code_col is None:
        log.warning(f"  {sheet}: country/code columns not found")
        return None

    # Estimate columns with valid years
    est_cols = [(y, s) for (y, s) in new_cols if s == "estimate" and y is not None]
    if not est_cols:
        log.warning(f"  {sheet}: no estimate columns parsed")
        return None

    pieces = []
    for col in est_cols:
        y = col[0]
        sub = data[[country_col, code_col, col]].copy()
        sub.columns = ["country", "iso3_wgi", "estimate"]
        sub["year"] = y
        sub["ind_code"] = ind_code
        pieces.append(sub)

    out = pd.concat(pieces, ignore_index=True)
    out["estimate"] = pd.to_numeric(out["estimate"], errors="coerce")
    out["country"]  = out["country"].astype(str).str.strip()
    out["iso3_wgi"] = out["iso3_wgi"].astype(str).str.strip().str.upper()
    out = out.dropna(subset=["country", "estimate"])
    out = out[out["country"].str.len() > 0]
    out = out[~out["country"].str.lower().isin({"nan", "none"})]
    return out

if not WGI_PATH.exists():
    log.warning(f"WGI file not found at {WGI_PATH}")
else:
    log.info(f"Reading WGI from {WGI_PATH}")
    xl = pd.ExcelFile(WGI_PATH)
    log.info(f"  sheets: {xl.sheet_names}")

    pieces = []
    for sheet, code in SHEET_TO_CODE.items():
        if sheet not in xl.sheet_names:
            log.warning(f"  expected sheet '{sheet}' missing; skipping {code}")
            continue
        df = parse_wgi_sheet(WGI_PATH, sheet, code)
        if df is not None:
            log.info(f"  {sheet} → {code}: {len(df):,} country-years")
            pieces.append(df)

    if not pieces:
        raise RuntimeError("Could not parse any WGI sheets.")

    wgi_long = pd.concat(pieces, ignore_index=True)

    # WGI's embedded Code column is mostly ISO3 but has a few legacy codes
    WGI_LEGACY_FIX = {"ADO": "AND", "ROM": "ROU", "ZAR": "COD", "TMP": "TLS",
                      "WBG": "PSE", "KSV": "XKX"}
    wgi_long["iso3"] = wgi_long["iso3_wgi"].map(WGI_LEGACY_FIX).fillna(wgi_long["iso3_wgi"])
    # Anything that isn't a clean 3-letter code → resolve from country name
    bad_mask = ~wgi_long["iso3"].str.fullmatch(r"[A-Z]{3}", na=False)
    if bad_mask.any():
        wgi_long.loc[bad_mask, "iso3"] = to_iso3(
            wgi_long.loc[bad_mask, "country"], source="WGI"
        )
    wgi_long = wgi_long.dropna(subset=["iso3"]).copy()
    wgi_long["year"] = pd.to_numeric(wgi_long["year"], errors="coerce").astype("Int64")
    wgi_long = wgi_long[(wgi_long["year"] >= YEAR_MIN) & (wgi_long["year"] <= YEAR_MAX)]
    wgi_long["year"] = wgi_long["year"].astype(int)

    wgi_wide = (wgi_long.pivot_table(index=["iso3", "year"], columns="ind_code",
                                     values="estimate", aggfunc="mean")
                .reset_index())

    wgi_wide.columns.name = None
    wgi_wide = wgi_wide.rename(columns={c: f"wgi_{c}" for c in wgi_wide.columns
                                         if c not in ("iso3", "year")})
    wgi_wide.to_csv(RAW/"wgi.csv", index=False)
    log.info(f"WGI saved: {len(wgi_wide):,} rows, {wgi_wide['iso3'].nunique()} countries, "
             f"indicators: {[c for c in wgi_wide.columns if c.startswith('wgi_')]}")
    print(wgi_wide.head())

In [15]:
raw = pd.read_excel(WGI_PATH, sheet_name="VoiceandAccountability", header=None)
print("Shape:", raw.shape)
print("\nFirst 18 rows, first 12 cols:")
print(raw.iloc[:18, :12].to_string())

FileNotFoundError: [Errno 2] No such file or directory: '/content/data/raw/wgi/wgidataset.xlsx'

In [ ]:
import requests
url = "https://www.worldbank.org/content/dam/sites/govindicators/doc/wgidataset.xlsx"
dst = RAW/"wgi/wgidataset.xlsx"
dst.parent.mkdir(parents=True, exist_ok=True)
r = requests.get(url, timeout=60, headers={"User-Agent": "academic/1.0"})
r.raise_for_status()
dst.write_bytes(r.content)
print(f"WGI: {dst.stat().st_size:,} bytes")

## 7e. IMF AI Preparedness Index (AIPI)

Download from https://www.imf.org/external/datamapper/datasets/AIPI by clicking
**Export → Excel** on the datamapper. Save as
`/content/data/raw/imf_aipi/AIPI_country.xlsx`.

The file has one row per country with the overall AIPI plus four sub-indices
(Digital Infrastructure, Human Capital & Labor Market, Innovation & Economic
Integration, Regulation & Ethics). Cazzaniga et al. 2024 also publish AI
Exposure and Complementarity scores that may be in the same file - we keep
any columns that look like numeric indicators.

Snapshot year: ~2023. We assign year=2023 in the panel (single cross-section).


In [ ]:
AIPI_DIR = RAW/"imf_aipi"
AIPI_DIR.mkdir(parents=True, exist_ok=True)

AIPI_NAME_MAP = [
    (["digital", "infrastructure"], "digital_infra"),
    (["human", "capital"],          "human_capital"),
    (["innovation", "economic"],    "innovation"),
    (["innocation", "economic"],    "innovation"),
    (["regulation", "ethics"],      "regulation"),
    (["ai", "preparedness"],        "total"),
    (["preparedness", "index"],     "total"),
]

def aipi_label(filename: str):
    fl = filename.lower()
    for keys, label in AIPI_NAME_MAP:
        if all(k in fl for k in keys):
            return label
    return None

def read_aipi_indicator_csv(fpath: Path) -> pd.DataFrame:
    """Return a 2-col frame: country, value. One row per country, most-recent value."""
    df = pd.read_csv(fpath)
    df.columns = [str(c).strip() for c in df.columns]
    country_col = next((c for c in df.columns
                        if "country" in c.lower() or c.lower() == "economy"), df.columns[0])
    year_cols = [c for c in df.columns
                 if str(c).strip().isdigit() and 2010 <= int(str(c).strip()) <= 2030]
    out = df[[country_col] + year_cols].copy()
    out = out.rename(columns={country_col: "country"})
    # Drop blank/NaN country rows EARLY — these are the duplicates that blow up merges
    out["country"] = out["country"].astype(str).str.strip()
    out = out[out["country"].str.len() > 0]
    out = out[~out["country"].str.lower().isin({"nan", "none", ""})]
    # Most-recent value per row
    if year_cols:
        for c in year_cols:
            out[c] = pd.to_numeric(out[c], errors="coerce")
        sorted_years = sorted(year_cols, key=int, reverse=True)
        out["value"] = out[sorted_years].bfill(axis=1).iloc[:, 0]
    else:
        val_col = next((c for c in df.columns
                        if c.lower() not in ("country","year") and
                        pd.api.types.is_numeric_dtype(df[c])), None)
        out["value"] = df[val_col] if val_col else np.nan
    # CRITICAL: one row per country, drop nulls
    out = out[["country", "value"]].dropna(subset=["value"])
    out = out.groupby("country", as_index=False)["value"].mean()
    return out

csv_files = list(AIPI_DIR.glob("*.csv"))
xlsx_files = list(AIPI_DIR.glob("*.xlsx"))

frames = []
if csv_files:
    log.info(f"Found {len(csv_files)} AIPI CSV files in {AIPI_DIR}")
    for f in csv_files:
        lbl = aipi_label(f.name)
        if lbl is None:
            log.warning(f"  could not classify {f.name}, skipping")
            continue
        df = read_aipi_indicator_csv(f)
        df = df.rename(columns={"value": f"aipi_{lbl}"})
        log.info(f"  {f.name} -> aipi_{lbl}: {len(df)} countries")
        frames.append(df)
elif xlsx_files:
    fpath = xlsx_files[0]
    log.info(f"Reading combined AIPI from {fpath}")
    aipi = pd.read_excel(fpath)
    aipi.columns = [str(c).strip().lower().replace(" ", "_").replace("&", "and")
                    for c in aipi.columns]
    country_col = next((c for c in aipi.columns if "country" in c), aipi.columns[0])
    aipi = aipi.rename(columns={country_col: "country"})
    aipi["country"] = aipi["country"].astype(str).str.strip()
    aipi = aipi[aipi["country"].str.len() > 0]
    keep_num = [c for c in aipi.columns
                if c != "country" and pd.api.types.is_numeric_dtype(aipi[c])]
    aipi = aipi[["country"] + keep_num].rename(columns={c: f"aipi_{c}" for c in keep_num})
    # One row per country
    aipi = aipi.groupby("country", as_index=False).mean(numeric_only=True)
    frames.append(aipi)
else:
    log.warning(f"No AIPI files in {AIPI_DIR}. "
                f"Export from https://www.imf.org/external/datamapper/datasets/AIPI")

if frames:
    aipi_out = frames[0]
    for f in frames[1:]:
        # Sanity-check before each merge
        assert aipi_out["country"].is_unique, "duplicate countries in aipi_out"
        assert f["country"].is_unique, "duplicate countries in incoming frame"
        aipi_out = aipi_out.merge(f, on="country", how="outer")

    aipi_out["iso3"] = to_iso3(aipi_out["country"], source="IMF AIPI")
    aipi_out = aipi_out.dropna(subset=["iso3"]).copy()
    aipi_out["year"] = 2023
    keep = ["iso3", "year"] + [c for c in aipi_out.columns if c.startswith("aipi_")]
    aipi_out = aipi_out[keep].drop_duplicates(subset=["iso3","year"]).reset_index(drop=True)
    aipi_out.to_csv(RAW/"imf_aipi.csv", index=False)
    log.info(f"AIPI saved: {len(aipi_out)} countries, "
             f"{[c for c in aipi_out.columns if c.startswith('aipi_')]}")
    print(aipi_out.head())

In [ ]:
import shutil
src = Path("/content/data/raw/imf_aipi")
dst = RAW/"imf_aipi"
dst.mkdir(parents=True, exist_ok=True)
for f in src.glob("*.csv"):
    shutil.copy(f, dst/f.name)
    print(f"copied {f.name}")
print(f"\nNow in {dst}: {list(dst.glob('*.csv'))}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 7f. WB CLASS.xlsx — historical income classifications

Download `CLASS.xlsx` from the link inside
https://datahelpdesk.worldbank.org/knowledgebase/articles/906519. Save to
`/content/data/raw/wb_income/CLASS.xlsx`.

The "Country Analytical History" sheet has countries as rows and years as
columns, with cells containing the income-group code (L / LM / UM / H, with
some older codes like LMC/UMC). We melt it to long, standardize the codes,
and save a `iso3, year, income_group_hist` CSV. This *replaces* the time-
invariant current-classification merge with one that varies year-by-year.


In [ ]:
CLASS_PATH = RAW/"wb_income/CLASS.xlsx"
CLASS_PATH.parent.mkdir(parents=True, exist_ok=True)

if CLASS_PATH.exists():
    log.info(f"Reading WB CLASS.xlsx from {CLASS_PATH}")
    xl = pd.ExcelFile(CLASS_PATH)
    log.info(f"CLASS sheets: {xl.sheet_names}")

    sheet = next((s for s in xl.sheet_names if "analytical" in s.lower()), xl.sheet_names[0])
    log.info(f"  using sheet: {sheet}")

    # Header is usually a few rows in. Detect by scanning the first 10 rows.
    raw = pd.read_excel(CLASS_PATH, sheet_name=sheet, header=None)
    header_row = None
    for i in range(min(10, len(raw))):
        row_vals = [str(v).strip().lower() for v in raw.iloc[i].dropna().tolist()]
        if any("country" in v or "economy" in v for v in row_vals):
            header_row = i
            break
    if header_row is None:
        header_row = 0
    log.info(f"  header row: {header_row}")

    cls = pd.read_excel(CLASS_PATH, sheet_name=sheet, header=header_row)
    # Drop blank columns
    cls = cls.loc[:, ~cls.columns.astype(str).str.contains("Unnamed", case=False, na=False)]
    cls.columns = [str(c).strip() for c in cls.columns]

    # Find country column
    country_col = next((c for c in cls.columns if "country" in c.lower() or "economy" in c.lower()), None)
    if country_col is None:
        raise RuntimeError(f"No country column in CLASS. Columns: {list(cls.columns)}")

    # Year columns: anything that looks like a fiscal year (e.g. "1987", "FY24", "1987-88")
    def is_year_col(c):
        s = str(c).strip()
        # Match 4-digit year or "FYxx" or "yyyy-yy"
        if s.isdigit() and 1980 <= int(s) <= 2030:
            return True
        if s.lower().startswith("fy"):
            return True
        if "-" in s and s.split("-")[0].isdigit():
            return True
        return False

    year_cols = [c for c in cls.columns if is_year_col(c)]
    log.info(f"  detected {len(year_cols)} year columns: {year_cols[:5]}...{year_cols[-3:] if len(year_cols)>5 else ''}")

    long = cls.melt(id_vars=[country_col], value_vars=year_cols,
                    var_name="year_raw", value_name="ig_raw")

    # Standardize the year column
    def parse_year(s):
        s = str(s).strip()
        if s.isdigit():
            return int(s)
        if s.lower().startswith("fy"):
            yy = s[2:].split("-")[0]
            if yy.isdigit():
                yy = int(yy)
                return 2000 + yy if yy < 50 else 1900 + yy
        if "-" in s:
            head = s.split("-")[0]
            if head.isdigit():
                return int(head)
        return None

    long["year"] = long["year_raw"].apply(parse_year)
    long = long.dropna(subset=["year"]).copy()
    long["year"] = long["year"].astype(int)

    # Standardize income-group codes (older releases use LIC/LMC/UMC/HIC etc.)
    CODE_MAP = {
        "L":"L","LIC":"L","LOW":"L","LOW INCOME":"L",
        "LM":"LM","LMC":"LM","LOWER MIDDLE":"LM","LOWER-MIDDLE":"LM",
        "LMI":"LM","LOWER MIDDLE INCOME":"LM","LOWER-MIDDLE INCOME":"LM",
        "UM":"UM","UMC":"UM","UPPER MIDDLE":"UM","UPPER-MIDDLE":"UM",
        "UMI":"UM","UPPER MIDDLE INCOME":"UM","UPPER-MIDDLE INCOME":"UM",
        "H":"H","HIC":"H","HIGH":"H","HIGH INCOME":"H",
        "OECD":"H","NON-OECD":"H","HIGH-OECD":"H",
    }
    long["ig_clean"] = (long["ig_raw"].astype(str).str.strip().str.upper()
                        .map(CODE_MAP))
    long = long.dropna(subset=["ig_clean"]).copy()

    long["iso3"] = to_iso3(long[country_col], source="WB CLASS")
    long = long.dropna(subset=["iso3"]).copy()

    out = long[["iso3","year","ig_clean"]].rename(columns={"ig_clean": "income_group_hist"})
    out = out[(out["year"] >= YEAR_MIN) & (out["year"] <= YEAR_MAX)]
    out["mi_status_hist"] = out["income_group_hist"].isin(["LM","UM"]).astype(int)
    out["hi_status_hist"] = (out["income_group_hist"] == "H").astype(int)
    out = out.drop_duplicates(subset=["iso3","year"])
    out.to_csv(RAW/"income_groups_hist.csv", index=False)
    log.info(f"Income history saved: {len(out):,} country-year rows, "
             f"{out['iso3'].nunique()} countries, "
             f"{out['year'].min()}-{out['year'].max()}")
    print(out.head())
else:
    log.warning(f"CLASS.xlsx not found at {CLASS_PATH}. "
                f"Download from https://datahelpdesk.worldbank.org/knowledgebase/articles/906519")


In [ ]:
!find /content -iname "CLASS.xlsx" 2>/dev/null
!find /content -iname "*income*class*" 2>/dev/null

In [ ]:
import requests
url = "https://datacatalogfiles.worldbank.org/ddh-published/0037712/DR0090755/CLASS.xlsx"
dst = RAW/"wb_income/CLASS.xlsx"
dst.parent.mkdir(parents=True, exist_ok=True)
r = requests.get(url, timeout=60, headers={"User-Agent": "academic/1.0"})
r.raise_for_status()
dst.write_bytes(r.content)
print(f"downloaded {dst.stat().st_size:,} bytes to {dst}")

## 7g. ILOSTAT — KIS share (Knowledge-Intensive Services)

Pull employment by economic activity (ISIC Rev. 4) from ILOSTAT and compute the
**KIS share** = (employment in J + K + M) / (total employment), where:
- **J** = Information and communication
- **K** = Financial and insurance activities
- **M** = Professional, scientific, and technical activities

We use the ILOSTAT bulk-download endpoint. If the bulk URL fails for any
reason, the cell suggests a manual fallback. Annual indicator id:
`EMP_TEMP_SEX_ECO_NB_A`.

This replaces the GGDC 10-Sector / ETD route — those products either end in
~2010 (10-Sector) or 2018 with no advanced economies (ETD), so they don't fit
a 2010–2024 global panel.


In [ ]:
import gzip

ILOSTAT_INDICATOR = "EMP_TEMP_SEX_ECO_NB_A"
ILOSTAT_URLS = [
    # Bulk download endpoints (try in order)
    f"https://www.ilo.org/ilostat-files/WEB_bulk_download/indicator/{ILOSTAT_INDICATOR}.csv.gz",
    f"https://rplumber.ilo.org/data/indicator/?id={ILOSTAT_INDICATOR}&type=both&format=.csv",
]

log.info(f"Downloading ILOSTAT employment-by-activity ({ILOSTAT_INDICATOR})...")
ilo_raw = None
for u in ILOSTAT_URLS:
    try:
        r = requests.get(u, timeout=180, headers={"User-Agent": "academic-research/1.0"})
        r.raise_for_status()
        content = r.content
        if u.endswith(".gz"):
            content = gzip.decompress(content)
        ilo_raw = pd.read_csv(BytesIO(content), low_memory=False)
        log.info(f"  got {len(ilo_raw):,} rows from {u}")
        break
    except Exception as e:
        log.warning(f"  failed {u}: {e}")

if ilo_raw is None:
    log.error("ILOSTAT download failed on all endpoints. "
              "Manual fallback: download "
              f"https://www.ilo.org/ilostat-files/WEB_bulk_download/indicator/{ILOSTAT_INDICATOR}.csv.gz "
              "and place at /content/data/raw/ilostat/EMP_TEMP_SEX_ECO_NB_A.csv.gz")
else:
    # Normalize column names
    ilo_raw.columns = [c.lower() for c in ilo_raw.columns]
    log.info(f"ILOSTAT columns: {list(ilo_raw.columns)}")

    # The bulk file has: ref_area, source, indicator, sex, classif1, time, obs_value, ...
    # classif1 looks like "ECO_ISIC4_TOTAL", "ECO_ISIC4_J", "ECO_ISIC4_K", etc.
    # We want sex=SEX_T (total), and the ISIC4 sector codes.
    ilo = ilo_raw.copy()

    # Filter to total sex
    if "sex" in ilo.columns:
        ilo = ilo[ilo["sex"].astype(str).str.upper().isin(["SEX_T", "TOTAL", "T"])]

    # Restrict to ISIC4 series only
    if "classif1" in ilo.columns:
        ilo = ilo[ilo["classif1"].astype(str).str.startswith("ECO_ISIC4_")]

    # Extract sector code (J, K, M, N, TOTAL, ...) from classif1
    ilo["sector"] = (ilo["classif1"].astype(str)
                     .str.replace("ECO_ISIC4_", "", regex=False))

    # Keep the four series we need
    KIS_SECTORS = ["J", "K", "M"]  # add "N" if you want a broader KIS definition
    keep_sectors = KIS_SECTORS + ["TOTAL"]
    ilo = ilo[ilo["sector"].isin(keep_sectors)].copy()

    # Standardize year and value
    year_col = "time" if "time" in ilo.columns else "year"
    val_col  = "obs_value" if "obs_value" in ilo.columns else "value"
    country_col = "ref_area" if "ref_area" in ilo.columns else "country"

    ilo["year"] = pd.to_numeric(ilo[year_col], errors="coerce").astype("Int64")
    ilo["emp"]  = pd.to_numeric(ilo[val_col],  errors="coerce")
    ilo = ilo.dropna(subset=["year", "emp"]).copy()
    ilo["year"] = ilo["year"].astype(int)
    ilo = ilo[(ilo["year"] >= YEAR_MIN) & (ilo["year"] <= YEAR_MAX)]

    # ref_area in ILOSTAT is already ISO3 (e.g. "USA", "GBR")
    ilo["iso3"] = ilo[country_col].astype(str).str.strip().str.upper()
    # Drop region aggregates (ILOSTAT uses things like "X01", "WORLD"); keep 3-letter codes
    ilo = ilo[ilo["iso3"].str.fullmatch(r"[A-Z]{3}")]

    # Pivot to one column per sector
    wide = (ilo.pivot_table(index=["iso3","year"], columns="sector",
                            values="emp", aggfunc="mean")
              .reset_index())
    wide.columns.name = None

    # Compute KIS share
    if "TOTAL" in wide.columns:
        kis_cols = [c for c in KIS_SECTORS if c in wide.columns]
        wide["emp_kis"] = wide[kis_cols].sum(axis=1, skipna=True)
        wide["kis_share"] = wide["emp_kis"] / wide["TOTAL"]
        # Sanity: anything > 1 or < 0 is bad
        bad = ((wide["kis_share"] < 0) | (wide["kis_share"] > 1)).sum()
        if bad:
            log.warning(f"  {bad} rows have implausible kis_share, setting to NaN")
            wide.loc[(wide["kis_share"] < 0) | (wide["kis_share"] > 1), "kis_share"] = np.nan

        out = wide[["iso3","year","kis_share"] + kis_cols + ["TOTAL"]].rename(
            columns={s: f"emp_{s.lower()}" for s in kis_cols + ["TOTAL"]})
        out.to_csv(RAW/"ilostat_kis.csv", index=False)
        log.info(f"ILOSTAT KIS saved: {len(out):,} rows, "
                 f"{out['iso3'].nunique()} countries, "
                 f"mean kis_share = {out['kis_share'].mean():.3f}")
        print(out.head())
    else:
        log.error("ILOSTAT data did not include a TOTAL sector; cannot compute KIS share.")


## 8. Merge everything we have into a single panel

In [ ]:
log.info("Assembling panel from available raw files...")

panel = None
# Note: files keyed on income_group MUST be merged AFTER income_groups_current.csv
#       has brought income_group into the panel. Likewise income_groups_hist
#       must come AFTER the panel has iso3/year keys.
files = [
    ("wdi_controls.csv",          ["iso3","year"]),
    ("pwt.csv",                   ["iso3","year"]),
    ("openalex_ai_pubs.csv",      ["iso3","year"]),
    ("ilostat_kis.csv",           ["iso3","year"]),
    ("wgi.csv",                   ["iso3","year"]),
    ("imf_aipi.csv",              ["iso3","year"]),
    ("income_groups_current.csv", ["iso3"]),
    ("income_groups_hist.csv",    ["iso3","year"]),
    ("gari_2024.csv",             ["iso3","year"]),
    ("ilo_rho_by_income.csv",     ["income_group"]),  # income-group level
]
for fname, keys in files:
    fpath = RAW/fname
    if not fpath.exists():
        log.warning(f"skip missing: {fname}")
        continue
    df = pd.read_csv(fpath)
    if panel is None:
        panel = df
        log.info(f"start with {fname}: panel shape = {panel.shape}")
        continue

    # If the merge key is missing from the panel, skip with a warning
    missing_keys = [k for k in keys if k not in panel.columns]
    if missing_keys:
        log.warning(f"skip {fname}: panel missing key(s) {missing_keys}")
        continue

    # Use left merge for one-shot lookups (income group, GARI snapshot, AIPI snapshot),
    # outer for things that should expand the panel.
    one_shot = fname in {
        "ilo_rho_by_income.csv",
        "income_groups_current.csv",
        "income_groups_hist.csv",
        "gari_2024.csv",
        "imf_aipi.csv",
    }
    how = "left" if one_shot else "outer"
    panel = panel.merge(df, on=keys, how=how)
    log.info(f"merged {fname}: panel shape = {panel.shape}")

# Useful derived columns
if "gdp_pc_ppp" in panel.columns:
    panel["log_gdp"] = np.log(panel["gdp_pc_ppp"])

# AI publications -> z-score as a starter G-index pillar
if "ai_pubs_count" in panel.columns:
    panel["ai_pubs_count"] = pd.to_numeric(panel["ai_pubs_count"], errors='coerce')
    panel["ai_pubs_log"] = np.log1p(panel["ai_pubs_count"])
    mu, sd = panel["ai_pubs_log"].mean(), panel["ai_pubs_log"].std()
    panel["G_pillar_activity_starter"] = (panel["ai_pubs_log"] - mu) / sd

try:
    panel.to_parquet(PROCESSED/"panel.parquet", index=False)
    log.info(f"Panel saved as parquet: {PROCESSED/'panel.parquet'}")
except ImportError:
    panel.to_csv(PROCESSED/"panel.csv", index=False)
    log.info(f"Panel saved as CSV: {PROCESSED/'panel.csv'}")

print(f"\nFinal panel: {panel.shape[0]:,} rows x {panel.shape[1]} cols")
print(f"Countries: {panel['iso3'].nunique() if 'iso3' in panel else 'NA'}")
print(f"Years: {panel['year'].min() if 'year' in panel else 'NA'} - "
      f"{panel['year'].max() if 'year' in panel else 'NA'}")
print(f"\nColumns available: {list(panel.columns)}")


In [ ]:
%whos DataFrame

In [ ]:
panel.to_csv("/content/final_panel.csv", index=False)

from google.colab import files
files.download("/content/final_panel.csv")

In [ ]:
# Pick the format you want:
df.to_csv("/content/final_panel.csv", index=False)
df.to_parquet("/content/final_panel.parquet")          # smaller, preserves dtypes
df.to_excel("/content/final_panel.xlsx", index=True)  # if you need Excel
from google.colab import files
files.download("/content/final_panel.csv")

## 9. Manual downloads — get these in your browser, then upload to Colab

After you download these files, upload them to `/content/data/raw/<subdir>/`
using the folder panel on the left (right-click → Upload). Then re-run
Section 8 to merge.

| Source | URL | Filename | Place in |
|--------|-----|----------|----------|
| **Oxford GARI 2024** | https://oxfordinsights.com/ai-readiness/ | `GARI_2024.xlsx` | `/content/data/raw/oxford_gari/` |
| **IMF AIPI** | https://www.imf.org/external/datamapper/datasets/AIPI | `AIPI_country.xlsx` | `/content/data/raw/imf_aipi/` |
| **ILO WP 140** | https://www.ilo.org/publications/generative-ai-and-jobs-refined-global-index-occupational-exposure | `ILO_WP140_exposure.xlsx` | `/content/data/raw/ilo_wp140/` |
| **GGDC 10-Sector** | https://www.rug.nl/ggdc/productivity/10-sector/ | `10SD_Database.xlsx` | `/content/data/raw/ggdc/` |
| **Findex (older waves)** | https://www.worldbank.org/en/publication/globalfindex/download-data | `GlobalFindex_<year>.xlsx` | `/content/data/raw/findex/` |
| **WGI** | https://www.worldbank.org/en/publication/worldwide-governance-indicators/interactive-data-access | `WGI_data.xlsx` | `/content/data/raw/wgi/` |
| **WB income history** | https://datahelpdesk.worldbank.org/knowledgebase/articles/906519 | `CLASS.xlsx` | `/content/data/raw/wb_income/` |
| **Stanford AI Index (optional)** | https://hai.stanford.edu/ai-index | `stanford_ai_index.csv` | `/content/data/raw/stanford_ai_index/` |

For each variable, see the table below for what column to extract.

In [ ]:
manual_table = pd.DataFrame([
    {"source": "Oxford GARI", "variable": "gari_government",
     "find_in_file": "Sheet 'Country Scores' column 'Government'"},
    {"source": "Oxford GARI", "variable": "gari_tech_sector",
     "find_in_file": "Sheet 'Country Scores' column 'Technology Sector'"},
    {"source": "Oxford GARI", "variable": "gari_data_infra",
     "find_in_file": "Sheet 'Country Scores' column 'Data and Infrastructure'"},
    {"source": "IMF AIPI", "variable": "aipi_digital_infra",
     "find_in_file": "Column 'Digital Infrastructure'"},
    {"source": "IMF AIPI", "variable": "aipi_human_capital",
     "find_in_file": "Column 'Human Capital and Labor Market Policies'"},
    {"source": "IMF AIPI", "variable": "aipi_innovation",
     "find_in_file": "Column 'Innovation and Economic Integration'"},
    {"source": "IMF AIPI", "variable": "aipi_regulation",
     "find_in_file": "Column 'Regulation and Ethics'"},
    {"source": "ILO WP 140", "variable": "rho_gradient4",
     "find_in_file": "Country-level table, 'Share in Gradient 4 (%)'"},
    {"source": "GGDC 10-Sector", "variable": "sigma_kis",
     "find_in_file": "Construct: sum of FIRE+CSPS employment / total non-ag employment"},
    {"source": "Findex 2011/14/17/24", "variable": "F components",
     "find_in_file": "fin45_t_d, fin11_t_d, fin22_t_d, fin10_t_d (newer waves)"},
    {"source": "WGI", "variable": "institutions",
     "find_in_file": "Government Effectiveness Estimate"},
])
manual_table

## 10. Download your data back to your laptop

After all cells finish, your panel and all raw files are in `/content/data/`.
To save them:

1. Click the folder icon (📁) on the left sidebar
2. Right-click `data` folder → Download
3. Colab will zip it and download to your computer

Or run the cell below to make a single zip:

In [ ]:
import os, shutil
src = "/content/data"
assert os.path.isdir(src) and os.listdir(src), f"{src} is missing or empty"
zip_path = shutil.make_archive("/content/data_acquisition_output", "zip", src)
print(f"Created: {zip_path} ({os.path.getsize(zip_path)/1024/1024:.1f} MB)")
from google.colab import files
files.download("/content/data_acquisition_output.zip")

## 11. Next steps

Once you have downloaded the manual sources and uploaded them to the
corresponding `/content/data/raw/<subdir>/` folders, the F and G index
construction needs to run. The full index-construction code is in the
companion repository under `src/construct/`. The key functions to call are:

```python
from src.construct.fintech_index import construct_fintech_index
from src.construct.ai_index import construct_ai_index

# After parsing the manual Findex files into a single df with the four components:
F = construct_fintech_index(findex_long)

# After parsing GARI + AIPI into merged AI-component frame:
G = construct_ai_index(ai_components)
```

The companion `joint_digital_mit_colab.ipynb` then runs the regressions
on the merged panel.

In [ ]:
findex_all = pd.read_csv(RAW/"findex_all_waves_raw.csv", low_memory=False)
findex_all.columns = [c.lower().strip() for c in findex_all.columns]

# Find the year column
year_col = next((c for c in findex_all.columns if c == "year"), None)
print(f"Year column: {year_col}")
print(f"Waves present: {sorted(findex_all[year_col].dropna().unique())}")
print(f"Rows per wave:")
print(findex_all[year_col].value_counts().sort_index())

# Check which of the 4 F-components are present
F_COMPONENTS = ["fin45_t_d", "fin11_t_d", "fin22_t_d", "fin10_t_d"]
print(f"\nF-components present: {[c for c in F_COMPONENTS if c in findex_all.columns]}")
print(f"F-components MISSING: {[c for c in F_COMPONENTS if c not in findex_all.columns]}")

# For each wave, count non-null values for each component (this tells us if the component
# even exists in that wave — older waves use slightly different indicator codes)
print(f"\nNon-null counts per wave × component:")
for comp in F_COMPONENTS:
    if comp in findex_all.columns:
        cov = findex_all.groupby(year_col)[comp].apply(lambda x: x.notna().sum())
        print(f"  {comp}: {dict(cov)}")

In [ ]:
# Look for any column that might be "saved at FI"
sav_candidates = [c for c in findex_all.columns
                  if "sav" in c.lower() or "save" in c.lower()
                  or c in ("fin10", "fin14_d", "saved_fi_t_d", "saved_t_d")]
print(f"Save-related columns: {sav_candidates}")

# Coverage check for the most likely candidates
candidates_to_check = ["fin10", "fin14_d", "fin14", "saved_fi_t_d", "saved_t_d", "save_any_t_d"]
print(f"\nCoverage by wave for save candidates:")
for col in candidates_to_check:
    if col in findex_all.columns:
        cov = findex_all.groupby("year")[col].apply(lambda x: x.notna().sum())
        print(f"  {col}: {dict(cov)}")
    else:
        print(f"  {col}: not in file")

# Also check what the 'group' values look like (we need to filter to 'all'/'total')
print(f"\n'group' column values: {findex_all['group'].value_counts().to_dict()}")
print(f"'group2' column values: {findex_all['group2'].value_counts().to_dict()}")